# Stage 9a v2 — 5-fold validation with decoder depth = 2 (Kaggle T4)

The Stage 9a ablation matrix flagged **decoder depth = 2** as the only ablation that beat the d=3 control by ≥ 2σ on fold 0 (CER 0.5574 vs 0.5674, Δ = −0.0100, 2σ threshold = 0.0088).

This notebook validates that fold-0 result at full 5-fold scale.  Every other knob is locked to the Stage 9a headline recipe:

| Knob | Value | Source |
|---|---|---|
| Encoder | Conformer 4×, d=256, kernel=15 | Stage 1 v2 locked |
| Decoder | **Transformer 2× (was 3)**, d=256 | **ablation winner** |
| lambda_ctc | 0.3 | Stage 9a, ablation U-curve minimum |
| Epochs | 80 | Stage 9a, e120 within noise |
| Optimiser | AdamW lr=5e-4 wd=5e-2 | Stage 1 v2 locked |
| Scheduler | OneCycleLR warmup 5% | Stage 1 v2 locked |
| Augmentation | temporal+spatial (LandmarkAugment) | Stage 1 v2 locked |
| Seed | 42 | original headline |

**Variant name**: `stage9a_v2`  (distinct from `stage9a` headline → separate checkpoint files + results JSON).

**Pass/fail vs Stage 9a (0.4889 ± 0.070 full / 0.4775 ± 0.067 stripped)**:
- ✅ Headline pass: full mean ≤ 0.481 (≥1 pt better than Stage 9a).
- ✅ Stretch: full mean ≤ 0.470 (~2 pt better; would warrant a permanent recipe change).
- ❌ Fail: full mean within ±2σ of 0.4889 → fold-0 win was a noise spike; revert.

**Wall-clock estimate on Kaggle T4 (one session)**:

| Phase | Time |
|---|---|
| Skeleton cache (if absent — auto-reused if attached) | ~30 min |
| 5-fold sweep at d=2 (~45 min/fold) | ~3 h 45 min |
| Total | **~4 h 15 min** |

Fits comfortably in Kaggle's 9 h budget.

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk 'mediapipe>=0.10.0' scipy --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')

import torch
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## Cell 2 — Locate cache + manifest + prior results

In [ ]:
import os, glob, json, shutil
def _first(pattern):
    m = (glob.glob(f'/kaggle/working/**/{pattern}', recursive=True)
         + glob.glob(f'/kaggle/input/**/{pattern}',  recursive=True))
    return m[0] if m else None

CACHE_PATH        = _first('skeleton_features_t32.pt')
CV_MANIFEST_FOUND = _first('subject_cv5.json')
RESULTS_FOUND     = _first('stage9a_v2_results.json')

OUT_CACHE    = CACHE_PATH        or '/kaggle/working/skeleton_features_t32.pt'
OUT_MANIFEST = CV_MANIFEST_FOUND or '/kaggle/working/subject_cv5.json'
RESULTS_PATH = '/kaggle/working/stage9a_v2_results.json'
if RESULTS_FOUND and not os.path.exists(RESULTS_PATH):
    shutil.copy(RESULTS_FOUND, RESULTS_PATH)

CKPT_DIR = '/kaggle/working/checkpoints'
LOG_DIR  = '/kaggle/working/logs'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(LOG_DIR, exist_ok=True)

for label, path in [('cache', OUT_CACHE), ('manifest', OUT_MANIFEST), ('results', RESULTS_PATH)]:
    print(f'{label:<10s}: {path}  (exists={os.path.exists(path)})')

## Cell 3 — Config  (only one knob differs from the Stage 9a headline)

In [ ]:
import logging, random
import numpy as np
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage9a_v2.log'))])

from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

# Locked from Stage 9a.
T_NATIVE     = 32
UPSAMPLE     = 2
D_MODEL      = 256
N_LAYERS     = 4
N_HEADS      = 4
CONV_KERNEL  = 15
DROPOUT      = 0.2
BATCH_SIZE   = 32
LR_PEAK      = 5e-4
WEIGHT_DECAY = 5e-2
GRAD_CLIP    = 1.0
NUM_EPOCHS   = 80
WARMUP_PCT   = 0.05
SEED         = 42
FOLDS        = list(range(5))
LAMBDA_CTC   = 0.3
DEC_N_HEADS  = 4

# The ONE thing that changes from Stage 9a:
DEC_N_LAYERS = 2          # was 3 in stage9a; ablation winner on fold 0
VARIANT_NAME = 'stage9a_v2'

cfg = Config(
    data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english',
                    max_zips=None, max_frames=64, seed=SEED),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
                      lr=LR_PEAK, weight_decay=WEIGHT_DECAY,
                      grad_clip=GRAD_CLIP, num_workers=2,
                      warmup_pct=WARMUP_PCT, seed=SEED,
                      checkpoint_dir=CKPT_DIR),
).build()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f'Device         : {cfg.device}')
print(f'Variant        : {VARIANT_NAME}')
print(f'Decoder layers : {DEC_N_LAYERS}  (Stage 9a used 3)')
print(f'lambda_ctc     : {LAMBDA_CTC}')
print(f'Folds          : {FOLDS}  (5-fold subject CV)')

## Cell 4 — Build skeleton cache + manifest if missing

No-op if already present.  ~30 min on T4 if from scratch (only happens when no prior kernel is attached).

In [ ]:
if not os.path.exists(OUT_CACHE):
    from wita_v2.datasets.subject_splits import stream_and_index_with_subjects
    from wita_v2.datasets.skeleton_cache  import extract_skeleton_features
    print('Building skeleton cache from scratch...')
    samples = stream_and_index_with_subjects(cfg)
    extract_skeleton_features(samples=samples, out_path=OUT_CACHE,
                              T_native=T_NATIVE, dtype=torch.float16)
cache = torch.load(OUT_CACHE, map_location='cpu', weights_only=False)
print(f'cache: {len(cache["feats"])} clips, '
      f'detect_rate={cache.get("frame_detect_rate",0)*100:.1f}%')

if not os.path.exists(OUT_MANIFEST):
    from wita_v2.datasets.cv_splits import build_cv5_manifest, save_cv5_manifest
    fake = [(b'', cache['labels'][i], cache['subjects'][i])
            for i in range(len(cache['feats']))]
    manifest = build_cv5_manifest(fake, n_folds=5, seed=SEED)
    save_cv5_manifest(manifest, OUT_MANIFEST)
else:
    from wita_v2.datasets.cv_splits import load_cv5_manifest
    manifest = load_cv5_manifest(OUT_MANIFEST)
print(f'manifest: {manifest["n_subjects_total"]} signers, {manifest["n_folds"]} folds')

## Cell 5 — Run the 5-fold sweep at d=2

Per-fold ~45 min on T4 (slightly faster than d=3 because the decoder is shallower).  Resume-aware via stage9a_v2_results.json.

In [ ]:
from wita_v2.training.stage9_train   import train_one_fold
from wita_v2.datasets.cv_splits       import fold_indices
from wita_v2.datasets.skeleton_augment import LandmarkAugment

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        all_results = json.load(f)
    completed = {(r['fold'], r['variant']) for r in all_results}
    print(f'Resuming — {len(completed)} folds already complete.')
else:
    all_results = []
    completed = set()

train_aug = LandmarkAugment()

for fold in FOLDS:
    if (fold, VARIANT_NAME) in completed:
        print(f'[skip] fold {fold} done'); continue
    train_idx, val_idx = fold_indices(manifest, fold, cache['subjects'])
    result = train_one_fold(
        cache=cache, train_idx=train_idx, val_idx=val_idx, cfg=cfg,
        fold=fold, variant=VARIANT_NAME,
        num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr_peak=LR_PEAK,
        weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, dropout=DROPOUT,
        d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        conv_kernel=CONV_KERNEL, upsample=UPSAMPLE, warmup_pct=WARMUP_PCT,
        dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
        lambda_ctc=LAMBDA_CTC, transform=train_aug, seed=SEED,
        checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR,
    )
    summary = {k: v for k, v in result.items() if k != 'history'}
    all_results.append(summary)
    with open(RESULTS_PATH, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  fold {fold}: best CER = {result["best_val_cer"]:.4f}\n')
print(f'\n{len(all_results)}/{len(FOLDS)} folds done.')

## Cell 6 — Aggregate + verdict vs Stage 9a

In [ ]:
import numpy as np
from scipy.stats import wilcoxon
from wita_v2.reports.template.stripped_cohort import dual_cohort_summary

with open(RESULTS_PATH) as f:
    all_results = json.load(f)
by_fold = {r['fold']: r for r in all_results if r['variant'] == VARIANT_NAME}

# Stage 9a headline numbers (per-fold) for paired comparison.
STAGE9A_FOLD = {0: 0.5681, 1: 0.5345, 2: 0.5054, 3: 0.4374, 4: 0.3991}
STAGE9A_FULL_MEAN     = 0.4889
STAGE9A_STRIPPED_MEAN = 0.4775

print(' fold    Stage 9a    Stage 9a v2    Δ (v2 - 9a)    best ep')
for f in FOLDS:
    if f not in by_fold:
        print(f'  {f:>2d}    {STAGE9A_FOLD[f]:.4f}      pending'); continue
    r = by_fold[f]
    d = r['best_val_cer'] - STAGE9A_FOLD[f]
    tag = '✅' if d <= -0.0088 else ('❌' if d >= 0.0088 else '⚖')
    print(f'  {f:>2d}    {STAGE9A_FOLD[f]:.4f}      {r["best_val_cer"]:.4f}        '
          f'{d:+.4f} {tag}    ep {r["best_epoch"]}')

if len(by_fold) >= 2:
    s = dual_cohort_summary(RESULTS_PATH, VARIANT_NAME)
    print(f'\n  Stage 9a v2 full     : {s["full_mean"]:.4f} ± {s["full_std"]:.4f}')
    print(f'  Stage 9a v2 stripped : {s["stripped_mean"]:.4f} ± {s["stripped_std"]:.4f}')
    print(f'  Stage 9a (headline) full : {STAGE9A_FULL_MEAN:.4f}  stripped : {STAGE9A_STRIPPED_MEAN:.4f}')
    full_delta = s['full_mean'] - STAGE9A_FULL_MEAN
    print(f'  Δ full mean (v2 - headline): {full_delta:+.4f}')

    # Paired Wilcoxon (n=5 limit; effect-size eyeball)
    if len(by_fold) == 5:
        a = np.array([STAGE9A_FOLD[f]          for f in FOLDS])
        b = np.array([by_fold[f]['best_val_cer'] for f in FOLDS])
        try:
            W, p = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
            print(f'  Paired Wilcoxon: W={W:.2f}  p≈{p:.4f}  n={len(a)}')
        except ValueError as e:
            print(f'  Wilcoxon n/a: {e}')

    print('\n=== Stage 9a v2 verdict ===')
    if s['full_mean'] <= 0.470:
        print(f'  ✅ STRETCH ({s["full_mean"]:.4f} ≤ 0.470) — d=2 is a real ~2 pt improvement.')
        print('     Adopt as the new Stage 9 headline.')
    elif s['full_mean'] <= 0.481:
        print(f'  ✅ HEADLINE ({s["full_mean"]:.4f} ≤ 0.481) — d=2 is a real ~1 pt improvement.')
        print('     Worth adopting; document in the appendix.')
    elif abs(full_delta) <= 0.0088:
        print(f'  ⚖  TIE ({full_delta:+.4f} within ±2σ).')
        print('     Fold-0 ablation gain did NOT generalise. Keep Stage 9a d=3 as headline.')
    else:
        print(f'  ❌ REGRESS ({full_delta:+.4f}).')
        print('     d=2 underfits at full scale; the deeper d=3 decoder is correct.')

## Cell 7 — Per-signer scatter (Stage 9a v2 vs Stage 9a)

Below the diagonal = Stage 9a v2 wins that signer.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Stage 9a per-signer (from the headline run -- copy in or load from /kaggle/input).
S9A_PER_SIGNER = _first('stage9a_results.json')
if S9A_PER_SIGNER:
    with open(S9A_PER_SIGNER) as f:
        s9a_all = json.load(f)
    s9a_ps = {}
    for r in s9a_all:
        if r.get('variant') == 'stage9a':
            s9a_ps.update(r.get('best_per_signer_val_cer', {}) or {})
    v2_ps = {}
    for r in all_results:
        if r['variant'] == VARIANT_NAME:
            v2_ps.update(r.get('best_per_signer_val_cer', {}) or {})
    signers = sorted(set(s9a_ps) | set(v2_ps))
    DATASET_LIMIT = ['PHW', 'KIM']
    MODEL_HARD    = ['PJH','SYB','KJM','KNY','LKS','YMG']
    def _c(s):
        if s in DATASET_LIMIT: return '#7f7f7f'
        if s in MODEL_HARD:    return '#d62728'
        return '#1f77b4'
    xs = [s9a_ps.get(s, float('nan')) for s in signers]
    ys = [v2_ps.get(s, float('nan')) for s in signers]
    colors = [_c(s) for s in signers]
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    lo, hi = 0.2, 1.0
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='y = x')
    ax.scatter(xs, ys, c=colors, s=48, edgecolor='black', linewidths=0.4)
    for s, x, y in zip(signers, xs, ys):
        ax.annotate(s, (x, y), fontsize=6, alpha=0.7,
                    xytext=(3, 3), textcoords='offset points')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
    ax.set_xlabel('Stage 9a (d=3)'); ax.set_ylabel('Stage 9a v2 (d=2)')
    ax.set_title('Per-signer val CER  (below y=x → v2 wins)')
    ax.legend(frameon=False, loc='upper left')
    ax.grid(True, linestyle=':', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, 'stage9a_v2_per_signer.png'), dpi=140)
    plt.show()
else:
    print('Stage 9a per-signer JSON not attached -- skipping scatter.')

## Cell 8 — Commit kernel

Save Version → Save & Run All so these survive:
- `/kaggle/working/stage9a_v2_results.json`
- `/kaggle/working/checkpoints/stage9a_v2_fold*_best.pt`  (5 checkpoints — feed into Stage 9b next)
- `/kaggle/working/logs/stage9a_v2.log`
- `/kaggle/working/logs/stage9a_v2_per_signer.png`

If verdict is **HEADLINE** or **STRETCH**, the Stage 9b LM rescoring kernel should attach this kernel's checkpoints (not the original Stage 9a's).